In [0]:
df = spark.read.table("teams.sensorx.df_sx_800")


In [0]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import VectorAssembler
from pyspark.sql.functions import col


# Prepare features and label columns
feature_cols = [
    "payload_xrayController_filamentCurrent",
    "payload_xrayController_temperature",
    "payload_xrayController_tubeCurrent",
    "payload_xrayController_tubeVoltage",
    "payload_xrayController_onTime"
]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
df_features = assembler.transform(df).withColumn("label", col("payload_fault_state").cast("double"))

# Split the data into 80% training and 20% test
train_df, test_df = df_features.randomSplit([0.8, 0.2], seed=42)

lr = LogisticRegression(maxIter=10, regParam=0.3, elasticNetParam=0.8)

# Fit the model on the 80% training data
lrModel = lr.fit(train_df)

# Print the coefficients and intercept for logistic regression
print("Coefficients: " + str(lrModel.coefficients))
print("Intercept: " + str(lrModel.intercept))

In [0]:
predictions = lrModel.transform(test_df)
#display(predictions.select("prediction", "label", "features"))

In [0]:
from pyspark.ml.classification import LogisticRegression

# Extract the summary from the returned LogisticRegressionModel instance trained
# in the earlier example
trainingSummary = lrModel.summary

# Obtain the objective per iteration
objectiveHistory = trainingSummary.objectiveHistory
print("objectiveHistory:")
for objective in objectiveHistory:
    print(objective)

# Obtain the receiver-operating characteristic as a dataframe and areaUnderROC.
trainingSummary.roc.show()
print("areaUnderROC: " + str(trainingSummary.areaUnderROC))

print("precisionByLabel: " + str(trainingSummary.precisionByLabel))

# Set the model threshold to maximize F-Measure
fMeasure = trainingSummary.fMeasureByThreshold
maxFMeasure = fMeasure.groupBy().max('F-Measure').select('max(F-Measure)').head()
bestThreshold = fMeasure.where(fMeasure['F-Measure'] == maxFMeasure['max(F-Measure)']) \
    .select('threshold').head()['threshold']
lr.setThreshold(bestThreshold)

do horizon for the failiures - create a new column with failiure horizon based on some horizon H


convert timestamp to datetime in pandas
delta  - add as a feature


In [0]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from pyspark.sql.functions import col

# onehotencoding for property deviceID

device_id = "425d5fd3-d7df-4e85-ada5-6f09f8945000"  
df_filter = df.filter(col("properties_deviceId") == device_id)
df_pandas = df_filter.toPandas()
df_pandas.drop(['timeStamp', 'properties_deviceId', 'payload_fault_faultName', 'serialNumber', 'GeneratorType'], axis=1, inplace=True)

# --- Lag Feature Engineering ---
lags = [1, 2, 3] # add more lags

for lag in lags:
    df_pandas[f'payload_xrayController_filamentCurrent_lag{lag}'] = df_pandas['payload_xrayController_filamentCurrent'].shift(lag)
    df_pandas[f'payload_xrayController_temperature_lag{lag}'] = df_pandas['payload_xrayController_temperature'].shift(lag)
    df_pandas[f'payload_xrayController_tubeCurrent_lag{lag}'] = df_pandas['payload_xrayController_tubeCurrent'].shift(lag)
    df_pandas[f'payload_xrayController_tubeVoltage_lag{lag}'] = df_pandas['payload_xrayController_tubeVoltage'].shift(lag)

df_pandas.dropna(inplace=True)

# --- Chronological Train/Test Split (no random shuffling!) ---
split = int(len(df_pandas) * 0.8)
train, test = df_pandas.iloc[:split], df_pandas.iloc[split:]

features = [c for c in df_pandas.columns if c != 'payload_fault_state']
X_train, y_train = train[features], train['payload_fault_state']
X_test, y_test = test[features], test['payload_fault_state']
display(X_train)
display(y_train)

# --- Model ---
clf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
clf.fit(X_train, y_train)

# --- Evaluation ---
print(classification_report(y_test, clf.predict(X_test)))